# Snapdragon Fitness App — DLC Generation

> **Run this notebook INSIDE the `qairt_container` Docker container.**
> Start it with: `./Tools/qairt_docker/run_qairt_docker.sh`
> Then: `cd Generate_models && jupyter notebook --ip=0.0.0.0 --no-browser --allow-root`

This notebook generates two quantised INT8 DLC files:
- `Quant_yoloNas_s_320.dlc` — YOLO-NAS-S person detector (320×320)
- `hrnet_axis_int8.dlc` — HRNet-W32 pose estimator (256×192, 17 keypoints)

In [ ]:
# Cell 1 — Imports & paths
import os
import subprocess
import shutil
import glob

QAIRT_SDK = os.environ.get('QAIRT_SDK_ROOT', '/opt/qairt/2.46.0.260424')
ASSETS_DIR = '/workspace/app/src/main/assets'
os.makedirs(ASSETS_DIR, exist_ok=True)

SNPE_ONNX_TO_DLC  = f'{QAIRT_SDK}/bin/x86_64-linux-clang/snpe-onnx-to-dlc'
SNPE_DLC_QUANTIZE = f'{QAIRT_SDK}/bin/x86_64-linux-clang/snpe-dlc-quantize'

print(f'QAIRT_SDK      = {QAIRT_SDK}')
print(f'ASSETS_DIR     = {ASSETS_DIR}')
print(f'snpe-onnx-to-dlc exists: {os.path.isfile(SNPE_ONNX_TO_DLC)}')

In [ ]:
# Cell 2 — Download YOLO-NAS-S and export to ONNX
subprocess.run(['pip', 'install', 'super-gradients==3.7.1'], check=True)

from super_gradients.training import models
import torch

model = models.get('yolo_nas_s', pretrained_weights='coco')
model.eval()

dummy_input = torch.zeros(1, 3, 320, 320)
torch.onnx.export(
    model,
    dummy_input,
    '/tmp/yolonas_s.onnx',
    opset_version=11,
    input_names=['input'],
    output_names=['boxes', 'scores'],
    dynamic_axes={'input': {0: 'batch'}}
)
print('✓ YOLO-NAS-S ONNX exported to /tmp/yolonas_s.onnx')

In [ ]:
# Cell 3 — Convert YOLO-NAS ONNX → DLC, then quantize to INT8

# Step 3a: ONNX → unquantised DLC
subprocess.run([
    SNPE_ONNX_TO_DLC,
    '--input_network', '/tmp/yolonas_s.onnx',
    '--output_path',   '/tmp/yolonas_s.dlc',
    '--input_dim',     'input', '1,3,320,320',
], check=True)
print('✓ YOLO-NAS DLC (unquantised) created')

# Step 3b: Quantize to INT8 (axis quantization for HTP NPU)
# *** SET THIS PATH to your COCO val2017 image directory ***
COCO_PATH = '/local/coco/val2017'

calib_list_path = '/tmp/calib_yolo.txt'
images = list(glob.glob(f'{COCO_PATH}/*.jpg'))[:200]
if len(images) == 0:
    print(f'⚠  No images found at {COCO_PATH}. Set COCO_PATH or use AI Hub pre-converted DLC.')
else:
    with open(calib_list_path, 'w') as f:
        for img in images:
            f.write(img + '\n')
    print(f'✓ Calibration list: {len(images)} images')

    subprocess.run([
        SNPE_DLC_QUANTIZE,
        '--input_dlc',  '/tmp/yolonas_s.dlc',
        '--output_dlc', f'{ASSETS_DIR}/Quant_yoloNas_s_320.dlc',
        '--input_list', calib_list_path,
        '--axis_quant',
    ], check=True)
    print('✓ YOLO-NAS DLC quantized and saved to assets/')

In [ ]:
# Cell 4 — Clone HRNet and export to ONNX
import sys

if not os.path.exists('/tmp/hrnet'):
    subprocess.run([
        'git', 'clone',
        'https://github.com/HRNet/HRNet-Human-Pose-Estimation.git',
        '/tmp/hrnet'
    ], check=True)
    print('✓ HRNet repository cloned')
else:
    print('✓ HRNet repository already exists')

# Install HRNet requirements
subprocess.run(['pip', 'install', 'yacs', 'easydict'], check=True)

# Weights path — download manually and place here:
# https://drive.google.com/drive/folders/1nzM_OBV9LbAEA7HClC0chEyf_7ECDXYA
HRNET_WEIGHTS = '/tmp/hrnet/models/pytorch/pose_coco/pose_hrnet_w32_256x192.pth'

if not os.path.isfile(HRNET_WEIGHTS):
    print(f'⚠  Weights not found at {HRNET_WEIGHTS}')
    print('   Download pose_hrnet_w32_256x192.pth from HRNet Google Drive and place at the path above.')
else:
    sys.path.insert(0, '/tmp/hrnet/lib')
    # Export to ONNX
    subprocess.run([
        'python3', '/tmp/hrnet/tools/export_onnx.py',
        '--cfg', '/tmp/hrnet/experiments/coco/hrnet/w32_256x192_adam_lr1e-3.yaml',
        'TEST.MODEL_FILE', HRNET_WEIGHTS,
        'OUTPUT_DIR', '/tmp'
    ], check=True)
    print('✓ HRNet ONNX exported to /tmp/hrnet.onnx')

In [ ]:
# Cell 5 — Convert HRNet ONNX → DLC, then quantize to INT8
HRNET_ONNX = '/tmp/hrnet.onnx'

if not os.path.isfile(HRNET_ONNX):
    print(f'⚠  {HRNET_ONNX} not found. Complete Cell 4 first.')
else:
    subprocess.run([
        SNPE_ONNX_TO_DLC,
        '--input_network', HRNET_ONNX,
        '--output_path',   '/tmp/hrnet.dlc',
        '--input_dim',     'input', '1,3,256,192',
    ], check=True)
    print('✓ HRNet DLC (unquantised) created')

    # Reuse COCO calibration list from Cell 3
    calib_list_path = '/tmp/calib_yolo.txt'
    if not os.path.isfile(calib_list_path):
        print('⚠  Calibration list not found. Run Cell 3 first with COCO_PATH set.')
    else:
        subprocess.run([
            SNPE_DLC_QUANTIZE,
            '--input_dlc',  '/tmp/hrnet.dlc',
            '--output_dlc', f'{ASSETS_DIR}/hrnet_axis_int8.dlc',
            '--input_list', calib_list_path,
            '--axis_quant',
        ], check=True)
        print('✓ HRNet DLC quantized and saved to assets/')

In [ ]:
# Cell 6 — Verify outputs
print('=== DLC Asset Verification ===')
all_ok = True
for fname in ['Quant_yoloNas_s_320.dlc', 'hrnet_axis_int8.dlc']:
    path = os.path.join(ASSETS_DIR, fname)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) // (1024 * 1024)
        print(f'  ✓ {fname}  ({size_mb} MB)')
    else:
        print(f'  ❌ MISSING: {path}')
        all_ok = False

if all_ok:
    print('\n✅ All DLC assets ready. Proceed with Android build.')
else:
    print('\n⚠  Some assets are missing.')
    print('   Alternative: download pre-converted INT8 DLC from https://aihub.qualcomm.com')
    print('   (YOLO-NAS-S + HRNet-W32 for SM8750), rename and copy to app/src/main/assets/')